# ROUGH-PATH-FORECASTER Exploration Notebook

This notebook explores signature methods, kernel computations, and Log-ODE for ETF forecasting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from signature_core import SignatureComputer, NeumannSignatureKernel, AdaptiveDepthSelector
from kernel_engine import GaussianProcessForecaster, KernelRidgeForecaster
from log_ode import LogODEForecaster, RoughPathEstimator
from models import EnsembleForecaster

%matplotlib inline
sns.set_style('darkgrid')

## 1. Signature Computation Demo

In [ ]:
# Create a simple 2D path (e.g., price path)
path = np.array([
    [0, 0],
    [1, 0.5],
    [2, 0.8],
    [3, 0.6],
    [4, 1.2],
    [5, 1.5]
])

# Compute signature
computer = SignatureComputer(depth=3, lead_lag=True, basepoint=True, time_channel=True)
augmented = computer.augment_path(path)
signature = computer.compute_signature(path)

print(f"Original path shape: {path.shape}")
print(f"Augmented path shape: {augmented.shape}")
print(f"Signature terms: {len(signature)}")
print("\nFirst 10 signature terms:")
for i, (k, v) in enumerate(list(signature.items())[:10]):
    print(f"  {k}: {v:.6f}")

## 2. Neumann Signature Kernel

In [ ]:
# Generate multiple random paths
paths = [np.cumsum(np.random.randn(30, 3), axis=0) for _ in range(10)]

# Compute kernel matrix
kernel = NeumannSignatureKernel(depth=2, tile_size=15)
K = kernel.kernel_matrix(paths)

plt.figure(figsize=(8, 6))
plt.imshow(K, cmap='viridis')
plt.colorbar(label='Kernel value')
plt.title('Signature Kernel Matrix')
plt.show()

## 3. Adaptive Depth Selection

In [ ]:
selector = AdaptiveDepthSelector(depths=[2, 3, 4])
X_paths = [np.cumsum(np.random.randn(20, 4), axis=0) for _ in range(50)]
y_returns = np.random.randn(50)

best_depth = selector.select_depth(X_paths, y_returns)
print(f"Selected optimal depth: {best_depth}")

## 4. Rough Path Estimation

In [ ]:
estimator = RoughPathEstimator()

# Create paths with different roughness
smooth_path = np.linspace(0, 1, 100).reshape(-1, 1)
rough_path = np.cumsum(np.random.randn(100, 1) * 0.1, axis=0)

smooth_roughness = estimator.compute_roughness(smooth_path)
rough_roughness = estimator.compute_roughness(rough_path)

print(f"Smooth path roughness: {smooth_roughness:.6f}")
print(f"Rough path roughness: {rough_roughness:.6f}")

## 5. Ensemble Forecast

In [ ]:
# Synthetic training data
X_train = [np.cumsum(np.random.randn(20, 3), axis=0) for _ in range(100)]
y_train = np.random.randn(100, 5)

# Train ensemble
ensemble = EnsembleForecaster(depths=[2, 3, 4], weights=[0.2, 0.6, 0.2])
ensemble.fit(X_train, y_train)

# Predict
X_test = [np.cumsum(np.random.randn(20, 3), axis=0) for _ in range(10)]
predictions = ensemble.predict(X_test)

print(f"Predictions shape: {predictions.shape}")

## 6. ETF Selection Demo

In [ ]:
from selection import ETFSelector, MacroRegimeContext

tickers = ['TLT', 'LQD', 'HYG', 'GLD', 'SLV']
selector = ETFSelector(tickers, 'AGG')

# Simulated predictions
predicted_returns = np.array([0.005, 0.003, 0.012, 0.008, -0.002])
picks = selector.select_picks(predicted_returns)

for pick in picks:
    print(f"Rank {pick['rank']}: {pick['ticker']} - Return: {pick['predicted_return']*100:.2f}% - Conviction: {pick['conviction']:.1f}%")

## 7. Macro Regime Detection

In [ ]:
regime_detector = MacroRegimeContext()

macro_scenarios = [
    {'VIX': 15, 'HY_SPREAD': 0.02, 'T10Y2Y': 0.5},
    {'VIX': 22, 'HY_SPREAD': 0.045, 'T10Y2Y': 0.2},
    {'VIX': 30, 'HY_SPREAD': 0.07, 'T10Y2Y': -0.1},
    {'VIX': 18, 'HY_SPREAD': 0.03, 'T10Y2Y': -0.05},
]

for i, macro in enumerate(macro_scenarios):
    regime = regime_detector.get_regime(macro)
    print(f"Scenario {i+1}: VIX={macro['VIX']}, Spread={macro['HY_SPREAD']} -> {regime['regime_label']}")

## 8. Performance Visualization

In [ ]:
# Simulated backtest results
dates = pd.date_range('2024-01-01', '2024-12-31', freq='D')
strategy_returns = pd.Series(np.random.randn(len(dates)) * 0.01 + 0.0005, index=dates)
benchmark_returns = pd.Series(np.random.randn(len(dates)) * 0.008, index=dates)

strategy_equity = (1 + strategy_returns).cumprod()
benchmark_equity = (1 + benchmark_returns).cumprod()

plt.figure(figsize=(12, 6))
plt.plot(strategy_equity.index, strategy_equity.values, label='Strategy', linewidth=2)
plt.plot(benchmark_equity.index, benchmark_equity.values, label='Benchmark', linewidth=2)
plt.legend()
plt.title('Strategy vs Benchmark Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Return')
plt.grid(True, alpha=0.3)
plt.show()